# AgroSmart — treino do modelo de visão

Treina a rede que estima **quantos por cento da lavoura foram prejudicados** pela estiagem, a partir de imagens do talhão (HU11).

TCC — Engenharia de Computação, Universidade São Judas Tadeu, 2026.

---

## Antes de começar: ligue a GPU

**Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: GPU (T4)**

Sem GPU o treino leva horas; com a T4 gratuita, cerca de 20 minutos.

## O que este notebook faz

1. baixa o código do projeto e a base de imagens;
2. mede o **estimador clássico** (heurística de cor), para ter com o que comparar;
3. treina a rede;
4. gera a tabela e as figuras que entram no documento do TCC;
5. baixa os pesos para você colocar no `.env` do módulo de visão.

Leia [docs/VISAO.md](https://github.com/USJT2026TCC/SmartAgro/blob/main/docs/VISAO.md) para entender **como o percentual é calculado** — as três decisões dessa conta mudam quanto a apólice paga.

## 1. Confirmar a GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("SEM GPU.")
    print("Ambiente de execucao -> Alterar o tipo de ambiente de execucao -> GPU (T4).")
    print("Da para continuar sem GPU, mas o treino vai levar horas.")

## 2. Baixar o código do projeto

In [ ]:
import os
import sys
from pathlib import Path

# Baixa o codigo. Se ja foi baixado, so atualiza: apagar e clonar de novo
# apagaria tambem a base de imagens ja baixada em visao/dados/.
if Path("/content/SmartAgro/.git").exists():
    !git -C /content/SmartAgro pull --ff-only
else:
    !git clone --depth 1 https://github.com/USJT2026TCC/SmartAgro.git /content/SmartAgro

# Todas as celulas seguintes contam com a pasta visao/ como pasta de trabalho.
%cd /content/SmartAgro/visao
!pip install -q -e .

# O Colab so enxerga um pacote instalado com "pip install -e" depois de
# reiniciar o kernel. Em vez de pedir para reiniciar, o codigo entra direto no
# caminho de importacao: sys.path para estas celulas, PYTHONPATH para os
# comandos com "!".
SRC = "/content/SmartAgro/visao/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC

# Se uma tentativa anterior de importar falhou, o Python guardou em cache um
# pacote "visao" vazio: a PASTA visao/ do repositorio, que nao e o pacote.
# Sem limpar, o erro continua mesmo com o caminho corrigido.
for nome in [m for m in sys.modules if m == "visao" or m.startswith("visao.")]:
    del sys.modules[nome]

import visao.indice as indice

print("pacote carregado de:", indice.__file__)
print("classes:", indice.CLASSES)
print("pesos da gravidade:", indice.PESOS_PADRAO)


## 3. Baixar a base de imagens

**Fonte:** SUİÇMEZ, Ç.; YILMAZ, C.; KAHRAMAN, H. T. *UAV-Based Multispectral Maize Dataset for Water Stress and Rust Detection*. Zenodo, 2025. DOI [10.5281/zenodo.19385720](https://doi.org/10.5281/zenodo.19385720)

**Licença:** CC BY 4.0 — uso livre, exigindo atribuição. Cite a fonte no TCC.

São 328 MB e 1.000 recortes de 224×224, metade de estresse hídrico e metade de ferrugem, cada um com a máscara de referência.

> A célula **confere o MD5** ao final. Um download truncado produz um ZIP que parece bom e falha na hora de extrair — aconteceu durante o desenvolvimento, e a conferência evita perder tempo procurando o erro no lugar errado.

In [ ]:
import hashlib
import shutil
from pathlib import Path

VISAO = Path("/content/SmartAgro/visao")
URL = (
    "https://zenodo.org/api/records/19385720/files/"
    "UAV-Based%20Multispectral%20Maize%20Dataset%20for%20Water%20Stress%20and%20Rust%20Detection%20"
    "(Water%20Stress%202025%20and%20Common%20Rust%202025)%20%E2%80%94%20Representative%20Subset%20(v1.0).zip/content"
)
MD5_ESPERADO = "e5a405830a703c9d4e73b159b0061cce"
TAMANHO_ESPERADO = 344_360_648

# Caminho ABSOLUTO: o arquivo vai sempre para visao/dados/, onde a secao 4 o
# procura, qualquer que seja a pasta atual do notebook.
destino = VISAO / "dados" / "milho-estresse-hidrico.zip"
destino.parent.mkdir(parents=True, exist_ok=True)

# Uma versao anterior deste notebook gravava o ZIP num caminho relativo, e ele
# podia parar fora de visao/. Se estiver la, move em vez de baixar 328 MB de novo.
for antigo in (
    Path("/content/SmartAgro/dados/milho-estresse-hidrico.zip"),
    Path("/content/dados/milho-estresse-hidrico.zip"),
):
    if not destino.exists() and antigo.exists() and antigo.stat().st_size == TAMANHO_ESPERADO:
        shutil.move(str(antigo), str(destino))
        print(f"ZIP encontrado em {antigo} e movido para {destino}")

if not destino.exists() or destino.stat().st_size != TAMANHO_ESPERADO:
    !wget -q --show-progress -O "{destino}" "{URL}"

resumo = hashlib.md5(destino.read_bytes()).hexdigest()

print(f"arquivo: {destino}")
print(f"tamanho: {destino.stat().st_size:,} bytes")
print(f"md5....: {resumo}")

if resumo != MD5_ESPERADO:
    raise RuntimeError(
        "Download incompleto ou corrompido. Apague o arquivo acima e rode esta celula "
        "de novo (nao use retomada de download)."
    )

print("\nArquivo integro. Fonte: Zenodo, DOI 10.5281/zenodo.19385720, CC BY 4.0.")


## 4. Preparar os dados

Extrai, traduz as classes para a ordem do AgroSmart e divide treino e validação.

A divisão é **espacial**, por coluna do ortomosaico, e não aleatória. Os recortes vêm de poucos voos sobre os mesmos talhões: sorteá-los um a um colocaria pedaços vizinhos da mesma lavoura nos dois lados, e o modelo seria avaliado sobre terra que já viu — com métrica boa e sem valor.

In [ ]:
%cd /content/SmartAgro/visao
!python treino/preparar_dados.py

from pathlib import Path

# Um comando com "!" que falha NAO interrompe o notebook: sem esta conferencia,
# o erro so apareceria na secao seguinte, apontando para o lugar errado.
indice_csv = Path("/content/SmartAgro/visao/dados/preparado/indice.csv")

if not indice_csv.exists():
    raise RuntimeError(
        "A preparacao nao gerou dados/preparado/indice.csv. Leia a mensagem logo acima. "
        "O mais comum e o ZIP nao estar em visao/dados/: rode a secao 3 de novo."
    )

print(f"\nPronto: {indice_csv}")


## 5. Medir o estimador clássico (a linha de base)

Antes de treinar, o número com o qual comparar. A heurística de cor separa solo de planta pelo excesso de verde e planta sadia de planta estressada pelo matiz — sem treino nenhum.

**Guarde esta saída para o TCC.** Ela é o que justifica o modelo treinado existir.

In [ ]:
%cd /content/SmartAgro/visao
!python treino/avaliar_baseline.py --divisao validacao --grupo water


## 6. Treinar

Cerca de 20 minutos em GPU T4, com 30 épocas.

Os pesos são gravados **na melhor época**, e o critério de "melhor" é o **erro do índice de dano**, não a IoU. IoU mede acerto pixel a pixel; o índice é o número que paga. Um modelo pode errar a borda de cada mancha e ainda acertar a proporção de lavoura afetada — e é a proporção que vira dinheiro.

Com uma ressalva que o próprio script aplica: **só qualifica a época que de fato previu as duas classes de estresse.** Um modelo que nunca marca estresse reporta dano zero em tudo e, num conjunto onde a maioria dos recortes tem pouco dano, ganharia um erro médio baixo por acidente — seria uma apólice que nunca paga, com boa métrica.

Se a sessão do Colab cair no meio, basta rodar esta célula de novo: os dados já estão preparados.


In [ ]:
%cd /content/SmartAgro/visao
!python treino/treinar.py --epocas 30 --lote 16 --versao visao-unet-1.0.0

from pathlib import Path

if not Path("/content/SmartAgro/visao/pesos/unet.pt").exists():
    raise RuntimeError("O treino nao gravou pesos/unet.pt. Leia a mensagem logo acima.")


## 7. A tabela do TCC

Compara a linha de base com o modelo treinado, no que importa: o erro do percentual de dano.

In [ ]:
%cd /content/SmartAgro/visao
import json
from pathlib import Path

historico = json.loads(Path("pesos/unet.historico.json").read_text(encoding="utf-8"))
melhor = min(historico["epocas"], key=lambda e: e["erro_medio_do_indice"])

print(f"Recortes de treino....: {historico['recortes_de_treino']}")
print(f"Recortes de validacao.: {historico['recortes_de_validacao']}")
print(f"Aumento de dados......: {'sim' if historico['aumento_de_dados'] else 'nao'}")
print(f"Melhor epoca..........: {melhor['epoca']} de {len(historico['epocas'])}")
print()
print(f"Erro medio do indice de dano: {melhor['erro_medio_do_indice']:.1%}")
print(f"IoU media...................: {melhor['iou_media']:.3f}")
print()
print("IoU por classe:")
for classe, valor in melhor["iou_por_classe"].items():
    print(f"  {classe:18} {valor:.3f}")
print()
print("Compare o erro do indice com o do estimador classico, medido na secao 5.")


In [ ]:
%cd /content/SmartAgro/visao
import matplotlib.pyplot as plt

epocas = [e["epoca"] for e in historico["epocas"]]
erro_indice = [e["erro_medio_do_indice"] for e in historico["epocas"]]
iou = [e["iou_media"] for e in historico["epocas"]]

figura, eixo = plt.subplots(figsize=(8, 4.5))
eixo.plot(epocas, erro_indice, label="erro medio do indice de dano", linewidth=2)
eixo.plot(epocas, iou, label="IoU media", linewidth=2, linestyle="--")
eixo.set_xlabel("epoca")
eixo.set_ylabel("valor")
eixo.set_title("Treino do segmentador de estresse hidrico")
eixo.grid(alpha=0.3)
eixo.legend()

figura.tight_layout()
figura.savefig("pesos/curva-de-treino.png", dpi=150)
plt.show()

print("Figura salva em pesos/curva-de-treino.png — serve para o documento do TCC.")


## 8. A figura qualitativa

Imagem, máscara de referência e predição, lado a lado. É o que mostra à banca o que o modelo faz — e também onde ele erra.

In [ ]:
%cd /content/SmartAgro/visao
import csv

import numpy as np
import torch
from matplotlib.colors import ListedColormap
from PIL import Image

from visao.indice import CLASSES, indice_de_dano
from visao.rede import carregar, padronizar

CORES = ListedColormap(["#8d6e63", "#2e7d32", "#fbc02d", "#c62828", "#6a1b9a"])
MAPA = {0: 0, 1: 2, 2: 3, 3: 1, 4: 4}  # base -> AgroSmart

modelo, versao = carregar("pesos/unet.pt")
raiz = Path("dados/preparado")

with (raiz / "indice.csv").open(encoding="utf-8") as arquivo:
    linhas = [l for l in csv.DictReader(arquivo) if l["divisao"] == "validacao" and l["grupo"] == "water"]

amostras = linhas[:: max(1, len(linhas) // 4)][:4]
figura, eixos = plt.subplots(len(amostras), 3, figsize=(9, 3 * len(amostras)))

for linha_do_grafico, registro in zip(eixos, amostras):
    imagem = Image.open(raiz / "images" / registro["imagem"]).convert("RGB")
    bruto = np.asarray(Image.open(raiz / "masks" / registro["mascara"]), dtype=np.int64)

    verdade = np.zeros_like(bruto)
    for origem, destino in MAPA.items():
        verdade[bruto == origem] = destino

    entrada = torch.from_numpy(np.asarray(imagem, dtype=np.float32) / 255.0).permute(2, 0, 1)
    entrada = padronizar(entrada)[None]  # a mesma padronizacao do treino
    with torch.no_grad():
        previsto = modelo(entrada).argmax(dim=1)[0].numpy()

    contar = lambda m: {nome: int((m == c).sum()) for c, nome in enumerate(CLASSES)}  # noqa: E731

    linha_do_grafico[0].imshow(imagem)
    linha_do_grafico[0].set_title("imagem", fontsize=10)
    linha_do_grafico[1].imshow(verdade, cmap=CORES, vmin=0, vmax=4)
    linha_do_grafico[1].set_title(f"referencia — dano {indice_de_dano(contar(verdade)):.0%}", fontsize=10)
    linha_do_grafico[2].imshow(previsto, cmap=CORES, vmin=0, vmax=4)
    linha_do_grafico[2].set_title(f"modelo — dano {indice_de_dano(contar(previsto)):.0%}", fontsize=10)

    for celula in linha_do_grafico:
        celula.axis("off")

figura.suptitle(
    "solo (marrom) · saudavel (verde) · estresse leve (amarelo) · severo (vermelho) · doenca (roxo)",
    fontsize=9,
)
figura.tight_layout()
figura.savefig("pesos/comparacao.png", dpi=150)
plt.show()

print(f"Modelo: {versao}")
print("Figura salva em pesos/comparacao.png")


## 9. Baixar os pesos

O arquivo tem cerca de 7 MB. Ele guarda apenas os números da rede, a versão e a lista de classes — nunca código.

In [ ]:
%cd /content/SmartAgro/visao
from google.colab import files

files.download("pesos/unet.pt")
files.download("pesos/unet.historico.json")
files.download("pesos/curva-de-treino.png")
files.download("pesos/comparacao.png")


## 10. Usar o modelo no AgroSmart

1. coloque `unet.pt` em `visao/pesos/` no seu computador;
2. no `visao/.env`:

```
VISAO_PESOS=pesos/unet.pt
```

3. rode o serviço:

```bash
.venv/Scripts/python -m visao servico --uma-vez
```

Ele vai anunciar na tela qual modelo carregou, e a versão sai **do arquivo de pesos**, não de uma variável — o que fica registrado com cada análise é o modelo que de fato rodou.

### O que muda no sistema

Com o modelo treinado, a confiança passa a vir **do próprio modelo**, e não de um valor fixo. Análises acima de 70% de confiança seguem direto para o oráculo; abaixo disso continuam indo ao perito (RF17).

Até aqui, com a heurística de cor, **toda** análise ia ao perito, porque a confiança dela é fixa em 45%. Isso era proposital: um número que ninguém conferiu não deve mover dinheiro.

### O que levar para o documento

| O que | De onde |
|---|---|
| Erro do estimador clássico | seção 5 |
| Erro do modelo treinado | seção 7 |
| Curva de treino | `curva-de-treino.png` |
| Comparação visual | `comparacao.png` |
| Citação da base | DOI 10.5281/zenodo.19385720, CC BY 4.0 |

Vale registrar também **o que o modelo não resolve**: a base é de milho, em outro país e outro solo. Aplicá-la à soja de São Simão é uma extrapolação, e o trabalho fica mais forte dizendo isso do que omitindo. A validação com imagens da lavoura real é trabalho futuro.